# Step 6 — Method fixes, fidelity vs training, and the soft mode at the right noise level

**Fixes from Step 5.**
1. *Birth jump is grid-sensitive* (36-pt vs 60-pt grids gave 0.27 vs 0.03 for (0,8)). Now: **bisect** between the last unimodal and first bimodal τ before measuring the jump.
2. *Network drift estimate is noisy* for overlapping classes. Now: 30k samples per τ and count-weighted Gaussian smoothing of the binned drift.
3. *Undertrained at the fold.* The fold sits at σ ≈ half·√τ_bif ≈ 3, in the tail of the EDM log-normal. Now: P_mean = −0.6, P_std = 1.4, 80 epochs, EMA, checkpoints at 20/40/60/80 → **fidelity vs training** curve.
4. *Soft-mode scan too low.* Now: σ from 0.1·half to 2·half, i.e. τ ∈ [0.01, 4].

Kaggle: GPU on, Internet on. ~35–40 min.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, math, copy
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['figure.dpi'] = 110
ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X_ALL = (ds.data.numpy().astype(np.float32)/127.5 - 1.0); Y_ALL = ds.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]; SIGMA_DATA = 0.5
CKPTS = [20, 40, 60, 80]

## Model and training (EMA, wider σ range, checkpoints)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]
        return x + self.c2(F.silu(self.n2(h)))

class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1)
        self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e)
        return self.out(self.u1(self.up(h2) + h1, e))

def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1)
    c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    c_in = 1/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(c_in*x, 0.25*torch.log(sigma))

def train(pair, epochs=80, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999):
    X = torch.from_numpy(X_ALL[np.isin(Y_ALL, pair)])[:, None].to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr)
    n = len(X); t0 = time.time(); ckpts = {}
    for ep in range(1, epochs+1):
        perm = torch.randperm(n, device=dev); tot = 0
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]
            sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev))
            xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0)
            w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2
            loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
        if ep % 10 == 0: print(f"  pair {pair} epoch {ep}/{epochs} loss {tot/(n//bs):.4f} ({time.time()-t0:.0f}s)")
        if ep in CKPTS:
            ckpts[ep] = copy.deepcopy(ema).eval()
            torch.save(ckpts[ep].state_dict(), f'/kaggle/working/unet_{pair[0]}{pair[1]}_ep{ep}.pt')
    return ckpts

NETS = {pair: train(pair) for pair in PAIRS}

## Analysis code with bisection and smoothing

In [ ]:
GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
TAUS = np.geomspace(0.1, 3.0, 36)

def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True)
    k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(np.convolve(h, k, mode='same'), 1e-300)

def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]:
                del pts[a:a+2]; ch = True; break
    return pts

def stationary_points(p):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx])

def zero_crossings(cen, mean, cnt, min_count=150):
    ok = cnt >= min_count; c, f = cen[ok], mean[ok]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0:
            out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)

def n_max(pts): return sum(k == 'max' for _, k in pts)

def birth_jump_bisect(get_pts, taus, iters=12):
    """get_pts(tau) -> stationary points. Bisect between last unimodal and first bimodal tau, then measure
    |separatrix at birth - stable fixed point just before birth|."""
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None:
            lo = tau; break
        if n_max(pts) == 1: hi = tau
    else:
        return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi); pts = get_pts(mid)
        if n_max(pts) >= 2: lo = mid
        else: hi = mid
    p_hi = get_pts(hi); p_lo = get_pts(lo)
    prev_max = [p for p, k in p_hi if k == 'max'][0]
    mx = sorted(p for p, k in p_lo if k == 'max'); mn = [p for p, k in p_lo if k == 'min']
    c = mn[0] if mn else 0.5*(mx[0]+mx[-1])
    return float(lo), float(abs(c-prev_max))

def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    mid = 0.5*(mu_a+mu_b); half = 0.5*float(delta @ u)
    return u.astype(np.float32), mid.astype(np.float32), half

@torch.no_grad()
def net_drift_points(net, Xflat, u, mid, half, tau, n=30000, nbins=70, smooth_bins=1.5):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1)
    b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins)
    k = np.exp(-0.5*(np.arange(-5, 6)/smooth_bins)**2)             # count-weighted Gaussian smoothing
    mean = np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
    return zero_crossings(0.5*(edges[:-1]+edges[1:]), mean, cnt)

def exact_points(Xflat, u, mid, half):
    m0 = (Xflat-mid) @ u / half
    return lambda tau: stationary_points(smooth_hist(m0, tau))

def balanced_flat(pair):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); return Xa, Xb, np.concatenate([Xa[:nmin], Xb[:nmin]])

## (a) Fidelity vs training: network fold vs exact fold along LDA (with bisection)

In [ ]:
print(f"{'pair':>7} {'exact tau':>10} {'exact jump':>11} | " + " | ".join(f"ep{e}: tau, jump" for e in CKPTS))
FID = {}
for pair in PAIRS:
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb)
    ex = birth_jump_bisect(exact_points(Xflat, u, mid, half), TAUS)
    row = []
    for ep in CKPTS:
        net = NETS[pair][ep]
        bj = birth_jump_bisect(lambda t: net_drift_points(net, Xflat, u, mid, half, t), TAUS)
        row.append(bj)
    FID[pair] = (ex, row)
    f = lambda bj: f"{bj[0]:.3f}, {bj[1]:.2f}" if bj else "  none   "
    print(f"{pair!s:>7} {f(ex):>22} | " + " | ".join(f"{f(bj):>16}" for bj in row))

# diagrams for the final checkpoint
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, pair in zip(axes, PAIRS):
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb); gp = exact_points(Xflat, u, mid, half)
    for tau in TAUS:
        for p, k in gp(tau): ax.plot(tau, p, '_', ms=9, color='k', alpha=0.4)
        for p, k in net_drift_points(NETS[pair][CKPTS[-1]], Xflat, u, mid, half, tau, n=15000): ax.plot(tau, p, 'o', ms=3, color='C0' if k == 'max' else 'C3')
    ax.set_xscale('log'); ax.set_xlabel('tau'); ax.axhline(0, color='k', lw=0.4); ax.set_title(f'{pair}: exact (grey) vs network ep{CKPTS[-1]} (colored)')
plt.tight_layout(); plt.show()

## (b) Soft mode across the transition (σ from 0.1·half to 2·half)

In [ ]:
@torch.no_grad()
def soft_mode(net, Xflat, sigma, n=512, iters=20, eps=1e-2):
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev)
    xt = (x0 + sigma*torch.randn_like(x0)).view(-1,1,28,28); s = torch.full((n,), sigma, device=dev)
    v = torch.randn(784, device=dev); v /= v.norm(); lam = 0.0
    for _ in range(iters):
        vv = v.view(1,1,28,28)
        Jv = ((denoise(net, xt+eps*vv, s) - denoise(net, xt-eps*vv, s))/(2*eps)).view(n, -1).mean(0)
        lam = float(Jv @ v); v = Jv/Jv.norm()
    return v.cpu().numpy().astype(np.float32), lam

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, pair in enumerate(PAIRS):
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb); net = NETS[pair][CKPTS[-1]]
    taus_s = np.geomspace(0.01, 4.0, 22); lams, ovl, vs = [], [], []
    for tau in taus_s:
        v, lam = soft_mode(net, Xflat, float(half*np.sqrt(tau))); lams.append(lam); ovl.append(abs(v @ u)); vs.append(v)
    ex = FID[pair][0]; tb = ex[0] if ex else 0.8
    axes[0, j].plot(taus_s, lams, 'o-', label='top eigenvalue of J'); axes[0, j].plot(taus_s, ovl, 's-', label='|v . u_LDA|')
    axes[0, j].axvline(tb, ls='--', color='k', lw=0.8); axes[0, j].set_xscale('log'); axes[0, j].set_xlabel('tau'); axes[0, j].set_title(f'{pair}: soft mode'); axes[0, j].legend(fontsize=8)
    k = int(np.argmin(np.abs(taus_s - tb))); v = vs[k]
    if v @ u < 0: v = -v
    for tau in TAUS:
        for p, kk in net_drift_points(net, Xflat, v, mid, half, tau, n=15000): axes[1, j].plot(tau, p, 'o', ms=3, color='C0' if kk == 'max' else 'C3')
    bjv = birth_jump_bisect(lambda t: net_drift_points(net, Xflat, v, mid, half, t), TAUS)
    axes[1, j].set_xscale('log'); axes[1, j].set_xlabel('tau'); axes[1, j].axhline(0, color='k', lw=0.4)
    axes[1, j].set_title(f'{pair}: drift along soft mode at tau={tb:.2f} (overlap {ovl[k]:.2f})\n' + (f'jump={bjv[1]:.2f} at tau={bjv[0]:.2f}' if bjv else 'never bimodal'), fontsize=9)
    print(f"{pair}: lambda max {max(lams):.3f} at tau={taus_s[int(np.argmax(lams))]:.2f}; overlap at tau_bif {ovl[k]:.2f}; soft-mode fold: {bjv}")
plt.tight_layout(); plt.show()

## (c) Re-run of the Step 4 census with bisection (CPU, exact diagrams, LDA, balanced)

In [ ]:
rows = []
for a, b in combinations(range(10), 2):
    Xa, Xb, Xflat = balanced_flat((a, b)); u, mid, half = lda_direction(Xa, Xb)
    bj = birth_jump_bisect(exact_points(Xflat, u, mid, half), np.geomspace(0.05, 3.0, 60))
    rows.append(((a, b), bj))
js = [bj[1] for _, bj in rows if bj]
print(f"bimodal: {len(js)}/45; jump median {np.median(js):.3f}, min {min(js):.3f}, max {max(js):.3f}; "
      f"jump<0.05: {sum(j < 0.05 for j in js)}, jump<0.1: {sum(j < 0.1 for j in js)}")
for (p, bj) in sorted(rows, key=lambda r: r[1][1] if r[1] else 9): print(p, f"tau={bj[0]:.3f} jump={bj[1]:.3f}" if bj else 'none')
plt.figure(figsize=(5, 3.4)); plt.hist(js, bins=np.linspace(0, 1.6, 33)); plt.xlabel('birth jump (bisected)'); plt.title('45 balanced MNIST pairs, LDA'); plt.tight_layout(); plt.show()

## What to report back

1. The fidelity table: does the network's (tau, jump) approach the exact values as training proceeds? At which checkpoint, if any, does it converge?
2. The soft-mode panels: does the top eigenvalue peak near the exact fold, and does the overlap with LDA rise there? What does the drift along the soft mode look like?
3. The bisected census: the corrected count of fold vs pitchfork-like pairs.